# [M3] XÂY MODEL AI SÂU HƠN BẰNG PYTORCH
## Buổi S04: Tự Viết Training Loop Chuẩn Mực Bằng PyTorch

---

### Metadata Khóa Học
* **Module:** M3 — XÂY MODEL AI SÂU HƠN BẰNG PYTORCH
* **Buổi:** S04
* **Thời lượng dự kiến:** 120 phút
* **Sản phẩm đầu ra (Session Output):** Notebook huấn luyện mạng nơ-ron với Training Loop và Validation Loop hoàn chỉnh
* **Mức độ liên quan VOAI / Olympic AI:** Làm chủ 100% luồng thực thi huấn luyện mô hình sâu trong phòng thi Olympic AI.

#### Mục tiêu học tập (Learning Objectives)
- Nắm vững như in 5 bước cốt tử của một vòng lặp huấn luyện PyTorch (Training Loop).
- Hiểu sự khác biệt sống còn giữa hai trạng thái mô hình: `model.train()` và `model.eval()`.
- Sử dụng đúng hàm mất mát (`nn.BCEWithLogitsLoss`) và thuật toán tối ưu hóa thích nghi (Adam Optimizer).
- Xây dựng vòng lặp kiểm chứng (Validation Loop) không tính gradient (`torch.no_grad()`) và tích lũy metric đánh giá chính xác qua các batch.

#### Điều kiện tiên quyết (Prerequisites)
- Hoàn thành các buổi M3_S01, M3_S02 và M3_S03.
- Nắm rõ khái niệm Epoch, Mini-Batch và Learning Rate.

---


## 1. WHY? — TẠI SAO PHẢI HỌC CHỦ ĐỀ NÀY?


Trong nhiều khóa học cấp tốc, học viên thường chỉ được dạy gọi các thư viện bọc sẵn (high-level wrappers) như Keras hay Trainer abstractions. Điều đó khiến bạn hoàn toàn bất lực khi bước vào phòng thi Olympic AI: Bạn không biết chèn thêm code đo lường metric tùy biến ở đâu, không biết cách clip gradient khi nổ dốc, và không hiểu tại sao mô hình của mình dự đoán sai trên tập validation.

Một kỹ sư AI đẳng cấp quốc tế **bắt buộc phải tự tay viết được một Training Loop chuẩn mực từ đầu đến cuối bằng PyTorch thuần**. Tự viết training loop trao cho bạn quyền kiểm soát 100% từng dòng dữ liệu, từng bước cập nhật trọng số và từng con số mất mát của mô hình.



## 2. CONCEPT — NỀN TẢNG LÝ THUYẾT & TRỰC GIÁC


### 1. 5 bước thần thánh của một Training Step trong PyTorch
Trong mỗi Mini-Batch của tập Train, bạn bắt buộc phải thực hiện tuần tự đúng 5 bước:
1. `optimizer.zero_grad()`: Xóa sạch gradient cũ còn sót lại từ batch trước.
2. `outputs = model(inputs)`: Lan truyền xuôi (Forward pass) để lấy dự đoán.
3. `loss = criterion(outputs, targets)`: Tính toán hàm mất mát.
4. `loss.backward()`: Lan truyền ngược (Backward pass) tự động tính gradient theo Chain Rule.
5. `optimizer.step()`: Cập nhật trọng số theo hướng tối ưu dốc.

### 2. `model.train()` vs `model.eval()`
* `model.train()`: Đưa mô hình vào chế độ huấn luyện. Bật tính năng Dropout (ngẫu nhiên tắt nơ-ron) và cập nhật thống kê động (running mean/var) của BatchNorm.
* `model.eval()`: Đưa mô hình vào chế độ đánh giá/kiểm thử. Tắt hoàn toàn Dropout (sử dụng 100% trọng số) và đóng băng BatchNorm để bảo đảm tính ổn định tuyệt đối của kết quả dự đoán.

### 3. Vòng lặp Kiểm chứng (Validation Loop)
* Luôn bọc toàn bộ validation trong khối `with torch.no_grad():` để:
  - Tiết kiệm bộ nhớ RAM/VRAM (không tốn chỗ lưu đồ thị tính toán).
  - Tăng tốc độ tính toán lên gấp 2-3 lần.



## 3. VISUAL INTUITION — TRỰC QUAN HÓA KIẾN THỨC


```
Cấu trúc Chu trình Training Loop hoàn chỉnh qua từng Epoch:
┌─────────────────────────────────────────────────────────────────────────────┐
│ FOR EPOCH in range(1, NUM_EPOCHS + 1):                                      │
│   ├── [PHASE 1: TRAINING] -> model.train()                                  │
│   │     For batch_X, batch_y in train_loader:                               │
│   │       1. zero_grad() -> 2. forward() -> 3. loss() -> 4. backward() -> 5. step()
│   │                                                                         │
│   └── [PHASE 2: VALIDATION] -> model.eval() with torch.no_grad():           │
│         For val_X, val_y in val_loader:                                     │
│           forward() -> compute val_loss -> accumulate val_accuracy          │
│                                                                             │
│   └── In báo cáo tiến độ: Epoch N | Train Loss: ... | Val Accuracy: ...     │
└─────────────────────────────────────────────────────────────────────────────┘
```



## 4. CODE IMPLEMENTATION & 5. WALKTHROUGH

*Toàn bộ code dưới đây được thiết kế thực tế, có thể chạy độc lập không phụ thuộc tải dữ liệu bên ngoài.*

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

# 1. Tao tap du lieu mau de huan luyen (400 mau, 5 dac trung, nhan nhi phan 0/1)
torch.manual_seed(42)
np.random.seed(42)

N = 400
X = torch.randn(N, 5)
# Tao nhan muc tieu co quy luat toan hoc + nhieu
true_weights = torch.tensor([[1.5], [-2.0], [0.8], [-1.2], [0.5]])
logits_true = X @ true_weights - 0.3
probs = torch.sigmoid(logits_true)
y = (probs > 0.5).float()

# Chia 80% Train (320 mau) va 20% Val (80 mau)
train_dataset = TensorDataset(X[:320], y[:320])
val_dataset = TensorDataset(X[320:], y[320:])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# 2. Xay dung kien truc mang nơ-ron
model = nn.Sequential(
    nn.Linear(5, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1) # Output Logits
)

# Khai bao Ham mat mat va Thuat toan toi uu Adam
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

print("Da chuan bi xong Du lieu, Mo hinh, Loss va Optimizer!")

### 🔍 Code Walkthrough (Khối 1)
Khối code chuẩn bị bài toán huấn luyện hoàn chỉnh:
- Sử dụng `TensorDataset` và `DataLoader` để quản lý phân chia Train/Validation theo Mini-batch.
- Mạng nơ-ron MLP xuất ra logits.
- `criterion = nn.BCEWithLogitsLoss()`: Tích hợp hàm mất mát Binary Cross-Entropy tối ưu số học.
- `optimizer = optim.Adam(..., lr=0.01)`: Thuật toán tối ưu hóa thích nghi hàng đầu hiện nay.


In [ ]:
# 3. Training & Validation Loop hoan chinh tu so 0
num_epochs = 20

print("===== BAT DAU HUAN LUYEN (TRAINING & VALIDATION LOOP) =====")
for epoch in range(1, num_epochs + 1):
    # --- PHASE 1: TRAINING ---
    model.train() # Bat che do huan luyen
    running_train_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()               # 1. Xoa gradient cu
        outputs = model(batch_X)            # 2. Forward pass
        loss = criterion(outputs, batch_y)  # 3. Tinh loss
        loss.backward()                     # 4. Backward pass
        optimizer.step()                    # 5. Cap nhat trong so
        
        running_train_loss += loss.item() * batch_X.size(0)
        
    epoch_train_loss = running_train_loss / len(train_loader.dataset)
    
    # --- PHASE 2: VALIDATION ---
    model.eval() # Bat che do danh gia
    running_val_loss = 0.0
    correct_preds = 0
    total_val_samples = 0
    
    with torch.no_grad(): # Khong tinh gradient de tiet kiem bo nho
        for val_X, val_y in val_loader:
            val_outputs = model(val_X)
            val_loss = criterion(val_outputs, val_y)
            running_val_loss += val_loss.item() * val_X.size(0)
            
            # Tinh do chinh xac (Accuracy): bien doi Logits -> Xac suat bang Sigmoid
            predicted_labels = (torch.sigmoid(val_outputs) >= 0.5).float()
            correct_preds += (predicted_labels == val_y).sum().item()
            total_val_samples += val_y.size(0)
            
    epoch_val_loss = running_val_loss / len(val_loader.dataset)
    epoch_val_acc = (correct_preds / total_val_samples) * 100
    
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch [{epoch:2d}/{num_epochs:2d}] | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.1f}%")

print("\n-> Huan luyen thanh cong! Vong lap chay muot ma khong loi.")

### 🔍 Code Walkthrough (Khối 2)
Khối code minh họa một Training & Validation Loop hoàn chỉnh đạt chuẩn thi đấu:
- Tách bạch rõ ràng 2 pha `model.train()` và `model.eval()`.
- Cài đặt đầy đủ 5 bước cập nhật trọng số trong pha huấn luyện.
- Tính toán chính xác độ mất mát trung bình theo số lượng mẫu thực tế (`* batch_X.size(0)`).
- Tính chỉ số độ chính xác (Validation Accuracy) bằng cách chuyển logits qua Sigmoid và so khớp với nhãn thực tế.


## 6. PRACTICE — BÀI TẬP THỰC HÀNH


### Bài 1 (Easy): Theo dõi lịch sử huấn luyện (History Tracking)
Tạo hai danh sách rỗng: `train_losses = []` và `val_losses = []`. Trong mỗi epoch, lưu giá trị mất mát vào danh sách tương ứng. Sau khi huấn luyện kết thúc, dùng Matplotlib vẽ đồ thị đường cong học tập (Learning Curves) so sánh Train Loss vs Val Loss.

### Bài 2 (Easy): Thử nghiệm với thuật toán SGD
Thay thế `optim.Adam` bằng `optim.SGD(model.parameters(), lr=0.05, momentum=0.9)`. So sánh tốc độ hội tụ của SGD có Momentum so với Adam.

### Bài 3 (Medium): Tích lũy Metric F1-Score trên tập Validation
Thay vì chỉ tính Accuracy, hãy import `f1_score` từ Scikit-Learn. Thu thập toàn bộ dự đoán `all_preds` và nhãn thực tế `all_targets` trên tập Validation, sau đó tính Macro-F1 ở cuối mỗi epoch.

### Bài 4 (Medium): Kỹ thuật Cắt tỉa Gradient (Gradient Clipping)
Trong các mạng nơ-ron sâu, gradient có thể bùng nổ quá lớn (Exploding Gradients). Hãy tìm hiểu hàm `torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)` và chèn nó vào giữa bước `loss.backward()` và `optimizer.step()`.

### Bài 5 (Challenge): Viết Lớp Trainer tổng quát
Đóng gói toàn bộ logic của Training và Validation Loop thành một class `PyTorchTrainer` có phương thức:
`fit(train_loader, val_loader, epochs=10)`
giúp tái sử dụng linh hoạt cho bất kỳ mô hình và bài toán nào trong tương lai.



## 7. VOAI CONNECTION — CHIẾN LƯỢC PHÒNG THI OLYMPIC


Trong phòng thi Olympic AI, nhiều thí sinh gặp hiện tượng kỳ lạ: Điểm số lúc kiểm thử thì rất cao nhưng khi nộp bài cho hệ thống chấm tự động thì bị lỗi `Time Limit Exceeded` hoặc kết quả nhảy loạn xạ. Nguyên nhân cốt lõi là thí sinh **quên gọi `model.eval()` và `with torch.no_grad():` trong hàm nộp bài**! Khi quên hai lệnh này, Dropout vẫn tiếp tục hoạt động làm kết quả mỗi lần chạy một khác, đồng thời PyTorch tiếp tục lưu đồ thị tính toán làm tràn bộ nhớ và kéo sập máy chủ chấm thi.



## 8. DEBUGGING CHALLENGE — BẮT LỖI THỰC CHIẾN


**Tình huống thực tế:** Thí sinh tính toán độ mất mát trung bình của epoch bằng cách cộng dồn trực tiếp `loss.item()` rồi chia cho số lượng batch:

```python
# CODE CÓ LỖI TÍNH SAI TRỌNG SỐ BATCH CUỐI:
# running_loss += loss.item()
# avg_loss = running_loss / len(train_loader)
# Neu batch cuoi cung co it mau hon (vi du chi co 8 mau thay vi 32 mau), 
# viec chia binh quan tren so batch se lam meo mo gia tri mat mat thuc te!
```

**Nhiệm vụ của bạn:**
1. Giải thích tại sao công thức chuẩn mực phải là nhân `loss.item()` với kích thước mẫu của batch hiện tại (`batch_X.size(0)`), sau đó mới chia cho tổng số mẫu `len(train_loader.dataset)`.
2. Sửa lại code để bảo đảm độ mất mát luôn được tính trung bình chuẩn xác trên từng mẫu dữ liệu.



## 9. KNOWLEDGE CHECK — KIỂM TRA ĐỘ HIỂU BẢN CHẤT


1. Thứ tự đúng đắn của 5 thao tác trong một bước huấn luyện (Training Step) của PyTorch là gì?
   - A. `backward()` -> `step()` -> `zero_grad()` -> `forward()` -> `loss()`.
   - B. `zero_grad()` -> `forward()` -> `loss()` -> `backward()` -> `step()`.
   - C. `forward()` -> `step()` -> `loss()` -> `backward()` -> `zero_grad()`.
   - D. `step()` -> `zero_grad()` -> `forward()` -> `loss()` -> `backward()`.

2. Lệnh `model.eval()` có tác dụng gì trong quá trình kiểm thử mô hình?
   - A. Xóa bỏ hoàn toàn mô hình khỏi bộ nhớ.
   - B. Chuyển mô hình sang chế độ đánh giá: Tắt cơ chế Dropout và đóng băng thống kê BatchNorm để cho ra dự đoán ổn định, có tính tất định.
   - C. Tự động tính toán điểm F1-score.
   - D. Tăng số lượng tham số của mạng.

3. Tại sao chúng ta bắt buộc phải gọi `loss.backward()` TRƯỚC KHI gọi `optimizer.step()`?
   - A. Vì `optimizer.step()` cần đọc các giá trị đạo hàm đã được tính và lưu trữ trong thuộc tính `.grad` của các tham số để biết hướng cập nhật.
   - B. Để xóa sạch dữ liệu của batch hiện tại.
   - C. Vì nếu gọi ngược lại sẽ gây lỗi cú pháp Python.
   - D. Vì hàm backward làm giảm kích thước ma trận.

4. Đoạn code `loss.item()` khác gì so với biến `loss` thông thường?
   - A. `loss.item()` trả về một số thực Python thông thường và ngắt kết nối khỏi đồ thị tính toán, giúp giải phóng bộ nhớ.
   - B. `loss.item()` là một tensor đa chiều.
   - C. `loss.item()` chạy trên card GPU.
   - D. Không có sự khác biệt nào.

5. Việc quên gọi lệnh `optimizer.zero_grad()` ở đầu mỗi batch sẽ dẫn đến hậu quả gì?
   - A. Mô hình sẽ bị dừng chạy ngay lập tức.
   - B. Gradient của batch hiện tại sẽ bị cộng dồn với gradient của tất cả các batch trước đó, khiến bước cập nhật trọng số trở nên khổng lồ và sai lệch hướng tối ưu.
   - C. Tập dữ liệu bị xóa mất nhãn.
   - D. Tốc độ nạp dữ liệu bị chậm lại.



## 10. SESSION SUMMARY — TỔNG KẾT BUỔI HỌC


* **What I learned:**
  - Nằm lòng 5 bước thần thánh của một chu trình huấn luyện PyTorch.
  - Phân định rạch ròi vai trò của `model.train()` và `model.eval()`.
  - Cách thiết lập vòng lặp kiểm chứng (Validation Loop) an toàn với `torch.no_grad()`.
  - Theo dõi và tính toán độ mất mát cũng như độ chính xác chuẩn xác qua từng Epoch.
* **What I can now do:**
  - Tự tay viết trọn vẹn một Training Loop từ đầu đến cuối mà không cần dựa dẫm vào các thư viện bọc ngoài.
  - Tự tin huấn luyện mạng nơ-ron sâu trên bất kỳ tập dữ liệu nào.
* **What I still need to practice:**
  - Kiểm soát hiện tượng quá khớp (Overfitting), lưu vết mô hình (Checkpointing) và xuất file nộp bài ở Buổi 5.



## 11. NEXT SESSION — BƯỚC ĐI TIẾP THEO


Buổi tiếp theo (**M3_S05**) là buổi tổng kết đỉnh cao của Module 3: **Chống Overfitting, Learning Curves, Checkpoint & Save/Load Model**! Bạn sẽ hoàn thiện notebook PyTorch có training loop, xuất file Checkpoint `.pt` và chính thức nhận Chứng chỉ **Certificate of Deep Learning Practice**!



---

## 🎯 Syllabus Traceability (Đối chiếu Syllabus)

Buổi học này đóng góp trực tiếp vào mục tiêu của **M3**:
- Tự viết Training loop hoàn chỉnh (Forward, Loss, Backward, Optimizer) ✓
- Chế độ model.train() vs model.eval() & no_grad validation ✓
- Đo lường loss & metric tích lũy qua từng Mini-batch ✓

* **Session Deliverable:** Mã nguồn Training Loop & Validation Loop chuẩn mực bằng PyTorch thuần
* **Đảm bảo chuẩn Syllabus Lock:** Nội dung nằm đúng chuẩn 100%, không lan man, giải quyết trực tiếp năng lực cốt lõi.
